# Dia 8 — A linha 4242 ✍️

**Linux para OSINT — 21 dias** · baseado no curso de [cipher387](https://github.com/cipher387/linux-for-OSINT-21-day) (MIT)

📘 Texto completo do dia: [dia-08-vim](https://github.com/Ridd1kulusC0d3r/linux-para-osint-21-dias/tree/main/dias/dia-08-vim)

## 🎬 Missão

> Às 18h a perícia entrega um presente envenenado: o log do servidor da
> PixVelox, **cinco mil linhas**, copiado de uma máquina que caiu antes de ser
> desligada. Você tenta abrir no bloco de notas do Colab; a aba engasga.
>
> A Mestra Iara deixa um bilhete na sua mesa e some de novo:
> *"Linha 4242. Ele troca todo `_` por `#MARE#`. Não salve no original."*
>
> Bento olha o arquivo por cima do seu ombro:
> — Vim. Abre, `:4242`, Enter. Só não me pergunte como sair.
> — Como eu saio?
> — Eu disse para não perguntar.

**Sua tarefa:** abrir `desafio/servidor-pixvelox.log` no Vim, ir até a linha
4242, desfazer o disfarce do maré_alta com uma substituição global e extrair
a flag `VAZANTE{...}`.

## ⚙️ Preparação

Rode a célula abaixo **uma vez**. Ela baixa o repositório para dentro do Colab e define a variável `L4O_HOME`, que todas as outras células usam.

> No Colab você é `root` e o `apt` funciona. Cada célula `%%bash` roda num processo separado: variáveis e `cd` não passam de uma célula para outra.

In [ ]:
import os, subprocess
from pathlib import Path

# Se já estamos dentro do repositório (Codespaces, máquina local), usa ele.
aqui = Path.cwd()
candidatos = [aqui, *aqui.parents]
raiz = next((p for p in candidatos if (p / 'lib' / 'comum.sh').exists()), None)

if raiz is None:  # Colab: clona o repositório
    raiz = Path('/content/linux-para-osint-21-dias')
    if not raiz.exists():
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Ridd1kulusC0d3r/linux-para-osint-21-dias', str(raiz)], check=True)

os.environ['L4O_HOME'] = str(raiz)
os.environ['L4O_CORES'] = '1'
print('Repositório em', raiz)

### Instalando e conferindo o Vim

```text
O Vim quase sempre já vem instalado (às vezes numa versão "mínima", vim-tiny).
`precisa vim` instala o pacote completo se o comando não existir.
`vim --version | head -n 1` mostra a versão sem abrir o editor.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-08-vim" && source "$L4O_HOME/lib/comum.sh"
titulo "Vim instalado?"
precisa vim
vim --version | head -n 1

### Abrindo um arquivo (modo interativo)

```text
`vim arquivo` abre o editor. Ele começa no MODO NORMAL: as teclas viram
comandos, não texto. Os modos que você precisa hoje:
  NORMAL   (Esc)  navegar e dar comandos (dd, yy, p, /busca)
  INSERÇÃO (i)    digitar texto como num bloco de notas
  COMANDO  (:)    linha no rodapé: :w salva, :q sai, :wq salva e sai, :q! sai SEM salvar
Num script ou no Colab o Vim "prende" a célula — por isso usamos `interativo`.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-08-vim" && source "$L4O_HOME/lib/comum.sh"
titulo "Abrindo o log no Vim"
interativo vim desafio/servidor-pixvelox.log
dica "Dentro do Vim: Esc e depois :q! para sair sem salvar. Respire. Funciona sempre."

### Quantas linhas tem esse log?

```text
Antes de mergulhar, meça o tamanho do problema. `wc -l` conta linhas.
Dentro do Vim, Ctrl+G mostra a mesma informação no rodapé.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-08-vim" && source "$L4O_HOME/lib/comum.sh"
titulo "Tamanho do log"
wc -l desafio/servidor-pixvelox.log
cat desafio/bilhete-iara.txt

### Pulando direto para uma linha: :N

```text
No Vim, digite :4242 e Enter (ou 4242G no modo normal). Para abrir o
arquivo JÁ na linha:  vim +4242 arquivo
Sem interface, o Vim tem o "modo ex silencioso" (-es): roda comandos de
dois pontos e sai. `+4242p` = vá para a linha 4242 e imprima (p = print).
`-cq` = execute o comando :q no fim. `ex` é o mesmo editor, só com outro nome.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-08-vim" && source "$L4O_HOME/lib/comum.sh"
titulo "Linha 4242 — três jeitos"
interativo vim +4242 desafio/servidor-pixvelox.log
vim -es +4242p -cq desafio/servidor-pixvelox.log
ex -s +'4242p' +'q!' desafio/servidor-pixvelox.log
sed -n '4242p' desafio/servidor-pixvelox.log      # o mesmo com sed (Dia 7)

### Buscando com /

```text
No modo normal, /texto procura para frente; n vai para a próxima ocorrência,
N volta. ?texto procura para trás. `:set hlsearch` destaca os resultados.
Equivalente não interativo: o comando :g/padrão/# lista as linhas que batem,
com o número na frente (igual a `grep -n`).
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-08-vim" && source "$L4O_HOME/lib/comum.sh"
titulo "Todas as linhas com mare_alta (só as 5 primeiras)"
interativo vim +/mare_alta desafio/servidor-pixvelox.log
vim -es -c 'g/mare_alta/#' -c 'q!' desafio/servidor-pixvelox.log | head -n 5
dica "São mais de 20 rascunhos. Quase todos são isca — confie no bilhete."

### Substituir em todo o arquivo: :%s/velho/novo/g

```text
  %   em todas as linhas
  s   substituir (substitute)
  g   todas as ocorrências da linha, não só a primeira
O maré_alta trocou '_' por '#MARE#'. Vamos desfazer NUMA CÓPIA (nunca no
original!) e salvar com :wq. O '#' não é especial no Vim, então vai direto.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-08-vim" && source "$L4O_HOME/lib/comum.sh"
titulo "Desfazendo a troca numa cópia"
mkdir -p treino
cp desafio/servidor-pixvelox.log treino/copia.log
interativo vim treino/copia.log
mostrar ":%s/#MARE#/_/g   e depois   :wq"
vim -es -c '%s/#MARE#/_/g' -c 'wq' treino/copia.log
vim -es +4242p -cq treino/copia.log
rm -r treino

### Apagar, copiar e colar: dd, yy, p

```text
  dd   apaga (recorta) a linha do cursor       3dd apaga 3 linhas
  yy   copia (yank) a linha do cursor          5yy copia 5 linhas
  p    cola abaixo do cursor                   P cola acima
  u    desfaz (undo)                           Ctrl+R refaz
Com `:normal` dá para mandar essas teclas pelo modo ex, sem interface.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-08-vim" && source "$L4O_HOME/lib/comum.sh"
titulo "dd, yy e p num arquivo de treino"
mkdir -p treino
cp desafio/treino-vim.txt treino/
interativo vim treino/treino-vim.txt
# 1G = vai para a linha 1; yy = copia; G = vai para o fim; p = cola
vim -es -c 'normal 1GyyGp' -c 'wq' treino/treino-vim.txt
# 2G = vai para a linha 2; dd = apaga
vim -es -c 'normal 2Gdd' -c 'wq' treino/treino-vim.txt
# e a substituição do próprio arquivo de treino
vim -es -c '%s/PixVeloz/PixVelox/g' -c 'wq' treino/treino-vim.txt
cat treino/treino-vim.txt
rm -r treino

### Salvar e sair: o resumo que salva vidas

```text
:w        salva                 :w novo.txt   salva com outro nome
:q        sai (se não mudou nada)
:wq ou :x salva e sai           ZZ (modo normal) faz o mesmo
:q!       sai DESCARTANDO tudo — o seu botão de pânico
Se o Vim travar com "swap file", alguém (talvez você) deixou o arquivo aberto:
o Vim criou um .swp. Escolha (R)ecover ou (D)elete e siga em frente.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-08-vim" && source "$L4O_HOME/lib/comum.sh"
titulo "Guia de sobrevivência"
printf '%-12s %s\n' "Esc" "volta para o modo normal (aperte duas vezes na dúvida)" \
  ":q!" "sai sem salvar" ":wq" "salva e sai" "i" "começa a digitar" \
  "/texto" "busca" ":4242" "vai para a linha 4242" "u" "desfaz"

### vimtutor: 30 minutos que valem o curso

```text
O próprio Vim traz um tutorial interativo, em português se o seu sistema
estiver em pt_BR. Faça pelo menos as lições 1 e 2.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-08-vim" && source "$L4O_HOME/lib/comum.sh"
titulo "Tutorial oficial"
interativo vimtutor
interativo vimtutor pt

## 🐍 Ponte Python

O mesmo resultado só com a biblioteca padrão do Python:

In [ ]:
!cd "$L4O_HOME/dias/dia-08-vim" && python3 equivalente.py

## 🧩 Desafio

Encontrou a flag? Troque o texto abaixo e rode a célula.

Dicas e passo a passo estão no README do dia.

In [ ]:
flag = 'VAZANTE{troque_aqui}'  # ✏️ cole sua resposta
!cd "$L4O_HOME/dias/dia-08-vim" && bash checar.sh "{flag}"

---

📝 Anote o que aprendeu no `DIARIO.md` do repositório.

Feito por [Deivison Lourenço](https://github.com/Ridd1kulusC0d3r) · curso original de [cipher387 / Cyber Detective](https://cybdetective.com)